# Lecture 6 — Class Exercise
## Choropleth Maps

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:
# Task 1 — 2007 life expectancy relative to the country-level global mean
# px.choropleth is appropriate for a world map with ISO-3 country identifiers.
gm_2007 = px.data.gapminder().query("year == 2007").copy()
global_mean = gm_2007["lifeExp"].mean()  # unweighted mean across countries
gm_2007["deviation"] = gm_2007["lifeExp"] - global_mean

# Identify the continent with the lowest mean deviation (rather than guessing).
continent_deviation = (gm_2007.groupby("continent", as_index=False)["deviation"]
                       .mean().sort_values("deviation"))
lowest_continent = continent_deviation.iloc[0]["continent"]
print(f"2007 country-level global average: {global_mean:.2f} years")
print(continent_deviation.to_string(index=False))

max_abs = gm_2007["deviation"].abs().max()
fig1 = px.choropleth(
    gm_2007, locations="iso_alpha", locationmode="ISO-3",
    color="deviation", hover_name="country",
    hover_data={"lifeExp": ":.1f", "deviation": ":+.1f", "iso_alpha": False},
    color_continuous_scale="RdBu", color_continuous_midpoint=0,
    range_color=(-max_abs, max_abs),
    labels={"lifeExp": "Life expectancy (years)",
            "deviation": "Difference from global mean (years)"},
    title=f"{lowest_continent} has the lowest average life expectancy relative to the 2007 global mean"
)
fig1.update_layout(margin=dict(l=0, r=0, t=65, b=0))
fig1.show()


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** [datasets/geo-countries](https://github.com/datasets/geo-countries), raw file: `https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson` (Natural Earth country boundaries). The code downloads the file when executed with internet access.

**Dataset:** Plotly Gapminder 2007, country population (`pop`) and ISO-3 codes (`iso_alpha`). Population is a numeric variable for each country.

**Chart type chosen:** `px.choropleth`, because this is a global country-level map and Plotly can draw the supplied GeoJSON polygons without needing a street-map basemap. `featureidkey` is inspected and validated against the GeoJSON properties rather than assumed.

**Colour scale chosen:** Sequential `Viridis`, because population is a non-negative magnitude, not a signed deviation from a meaningful zero. A logarithmic transform (`log10(pop)`) makes differences across several orders of magnitude readable; hover displays actual population.

**Insight:** The title identifies the country with the highest population **among countries in Gapminder 2007 that match the GeoJSON**, calculated from the data. Countries absent from Gapminder are not included.

**Execution note:** Task 2 requires an internet connection to download the external GeoJSON. If GitHub cannot be reached, download `countries.geojson` from the URL above and place it beside this notebook; the code will use the local file instead.


In [ ]:
# Task 2 — country populations with externally sourced GeoJSON
import os
import requests
import numpy as np

geojson_url = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"
local_path = "countries.geojson"
if os.path.exists(local_path):
    with open(local_path, "r", encoding="utf-8") as f:
        my_geojson = json.load(f)
else:
    try:
        response = requests.get(geojson_url, timeout=30)
        response.raise_for_status()
        my_geojson = response.json()
    except requests.RequestException as exc:
        raise RuntimeError(
            "GeoJSON download failed. Download countries.geojson from the source URL "
            "and save it next to this notebook, then rerun this cell."
        ) from exc

assert my_geojson.get("type") == "FeatureCollection"
assert my_geojson.get("features"), "GeoJSON contains no features"
print("First feature properties:", my_geojson["features"][0].get("properties", {}))

# Find an ISO-3 property by inspecting actual feature values and overlap with the dataset.
population = px.data.gapminder().query("year == 2007").copy()
iso_codes = set(population["iso_alpha"].dropna().astype(str))
props = my_geojson["features"][0].get("properties", {})
candidates = [k for k in props if any(term in k.lower() for term in ("iso", "adm0_a3", "sov_a3"))]
def matched_codes(key):
    return {str(f.get("properties", {}).get(key)) for f in my_geojson["features"]} & iso_codes
scores = {key: len(matched_codes(key)) for key in candidates}
if not scores or max(scores.values()) == 0:
    raise ValueError(f"No ISO-3 property matches Gapminder codes. Properties: {list(props)}")
geojson_key = max(scores, key=scores.get)
featureidkey = f"properties.{geojson_key}"
print("Chosen featureidkey:", featureidkey, "| matching countries:", scores[geojson_key])

# Keep only countries that match an actual GeoJSON feature.
population = population[population["iso_alpha"].isin(matched_codes(geojson_key))].copy()
assert not population.empty, "No matching countries between Gapminder and GeoJSON"
population["log10_population"] = np.log10(population["pop"])
largest = population.loc[population["pop"].idxmax()]

fig2 = px.choropleth(
    population, geojson=my_geojson,
    locations="iso_alpha", featureidkey=featureidkey,
    color="log10_population", color_continuous_scale="Viridis",
    hover_name="country",
    hover_data={"pop": ":,", "log10_population": False, "iso_alpha": False},
    labels={"log10_population": "Log10 population", "pop": "Population"},
    title=f"{largest['country']} has the largest population among mapped Gapminder countries (2007)"
)
fig2.update_geos(fitbounds="locations", visible=False)
fig2.update_layout(margin=dict(l=0, r=0, t=65, b=0))
fig2.show()
